# 04. 수익률 & 리스크 분석 (Return & Risk Analysis)

각 자산의 수익률 분포, 최대 낙폭, 변동성, Sharpe ratio 등 리스크 지표를 분석한다.

**지표**: 일간 수익률 분포, 누적 수익률, Maximum Drawdown, 연환산 변동성, Sharpe-like ratio, VaR

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from scipy import stats

from db import load_prices, get_asset_list

plt.rcParams['figure.figsize'] = (14, 6)
sns.set_theme(style='whitegrid')

In [ ]:
# 데이터 로드 및 일간 수익률 계산
assets = get_asset_list()

daily_returns = {}
daily_prices = {}

for _, row in assets.iterrows():
    source, symbol = row['source'], row['symbol']
    label = f"{symbol} ({source})"

    df = load_prices(symbol=symbol, source=source)
    if df.empty:
        continue

    # 일간 종가로 리샘플링
    daily = df.set_index('collected_at')['price'].resample('1D').last().ffill().dropna()
    if len(daily) < 5:
        continue

    daily_prices[label] = daily
    daily_returns[label] = daily.pct_change().dropna()

returns_df = pd.DataFrame(daily_returns).dropna()
prices_df = pd.DataFrame(daily_prices).dropna()

print(f"자산 수: {len(returns_df.columns)}")
print(f"공통 일수: {len(returns_df)}")
print(f"기간: {returns_df.index.min().date()} ~ {returns_df.index.max().date()}")

## 1. 일간 수익률 분포

In [ ]:
# 수익률 히스토그램 + KDE (자산별)
n_assets = len(returns_df.columns)
cols = min(3, n_assets)
rows = (n_assets + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 4*rows))
if n_assets == 1:
    axes = np.array([axes])
axes = axes.flatten()

for i, col in enumerate(returns_df.columns):
    ax = axes[i]
    data = returns_df[col].dropna()

    ax.hist(data, bins=50, density=True, alpha=0.6, edgecolor='black')
    data.plot.kde(ax=ax, color='red', linewidth=2)

    # 정규분포 오버레이
    x = np.linspace(data.min(), data.max(), 100)
    ax.plot(x, stats.norm.pdf(x, data.mean(), data.std()),
            'k--', linewidth=1, label='정규분포')

    # Shapiro-Wilk 검정 (최대 5000개 샘플)
    sample = data.sample(min(5000, len(data)), random_state=42)
    _, p_value = stats.shapiro(sample)

    ax.set_title(f'{col}\nShapiro p={p_value:.4f}')
    ax.set_xlabel('일간 수익률')
    ax.legend()

# 빈 서브플롯 숨기기
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('일간 수익률 분포 (KDE + 정규분포 비교)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 수익률 기술통계
desc = returns_df.describe().T
desc['skewness'] = returns_df.skew()
desc['kurtosis'] = returns_df.kurtosis()
desc.style.format('{:.6f}').set_caption('일간 수익률 기술통계')

## 2. 누적 수익률 (Cumulative Return)

In [ ]:
# 누적 수익률: 1원 투자 기준 성장 곡선
cumulative = (1 + returns_df).cumprod()

fig = go.Figure()
for col in cumulative.columns:
    fig.add_trace(go.Scatter(x=cumulative.index, y=cumulative[col], name=col))

fig.add_hline(y=1.0, line_dash='dash', line_color='gray',
              annotation_text='투자 원금 (1.0)')

fig.update_layout(
    title='누적 수익률 (1원 투자 기준)',
    xaxis_title='날짜', yaxis_title='성장 배수',
    height=500, hovermode='x unified'
)
fig.show()

## 3. 최대 낙폭 (Maximum Drawdown)

In [ ]:
def calc_drawdown(returns_series):
    """수익률 시리즈에서 드로다운을 계산한다."""
    cumulative = (1 + returns_series).cumprod()
    running_max = cumulative.cummax()
    drawdown = (cumulative - running_max) / running_max
    return drawdown, cumulative, running_max

# 모든 자산 드로다운 계산
drawdowns = {}
max_dd = {}

for col in returns_df.columns:
    dd, cum, rm = calc_drawdown(returns_df[col])
    drawdowns[col] = dd
    max_dd[col] = dd.min()

dd_df = pd.DataFrame(drawdowns)

In [ ]:
# Underwater Chart (드로다운 시각화)
fig = go.Figure()

for col in dd_df.columns:
    fig.add_trace(go.Scatter(
        x=dd_df.index, y=dd_df[col] * 100,
        name=f'{col} (Max: {max_dd[col]*100:.2f}%)',
        fill='tozeroy', opacity=0.6
    ))

fig.update_layout(
    title='드로다운 (Underwater Chart)',
    xaxis_title='날짜', yaxis_title='드로다운 (%)',
    height=500, hovermode='x unified'
)
fig.show()

## 4. 연환산 변동성 & Sharpe-like Ratio

In [ ]:
# 연환산 지표 계산
# 암호화폐: 365일, 주식/금: 252일
CRYPTO_SOURCES = {'UPBIT', 'BINANCE'}

risk_metrics = []

for col in returns_df.columns:
    ret = returns_df[col]
    
    # 소스 추출해서 연환산 일수 결정
    source_part = col.split('(')[-1].rstrip(')')
    ann_factor = 365 if source_part in CRYPTO_SOURCES else 252

    daily_mean = ret.mean()
    daily_std = ret.std()

    ann_return = daily_mean * ann_factor
    ann_vol = daily_std * np.sqrt(ann_factor)
    sharpe = ann_return / ann_vol if ann_vol > 0 else 0

    risk_metrics.append({
        '자산': col,
        '일 평균 수익률': f'{daily_mean*100:.4f}%',
        '일 변동성': f'{daily_std*100:.4f}%',
        '연환산 수익률': f'{ann_return*100:.2f}%',
        '연환산 변동성': f'{ann_vol*100:.2f}%',
        'Sharpe Ratio': f'{sharpe:.3f}',
        'Max Drawdown': f'{max_dd[col]*100:.2f}%',
        '연환산 일수': ann_factor
    })

risk_df = pd.DataFrame(risk_metrics)
risk_df.style.set_caption('리스크-수익 지표 종합 (무위험이자율 = 0%)')

## 5. Value at Risk (VaR)

In [ ]:
# Historical VaR (95%, 99%)
var_data = []

for col in returns_df.columns:
    ret = returns_df[col]
    var_95 = ret.quantile(0.05)   # 하위 5% = 95% VaR
    var_99 = ret.quantile(0.01)   # 하위 1% = 99% VaR
    cvar_95 = ret[ret <= var_95].mean()  # Conditional VaR (Expected Shortfall)

    var_data.append({
        '자산': col,
        'VaR 95%': f'{var_95*100:.4f}%',
        'VaR 99%': f'{var_99*100:.4f}%',
        'CVaR 95%': f'{cvar_95*100:.4f}%',
        '해석 (100만원 투자)': f'95% 확률로 일일 최대 손실 {abs(var_95)*1_000_000:,.0f}원'
    })

var_df = pd.DataFrame(var_data)
var_df.style.set_caption('Historical Value at Risk')

In [ ]:
# VaR 시각화: 수익률 분포에 VaR 선 표시
for col in returns_df.columns:
    ret = returns_df[col]
    var_95 = ret.quantile(0.05)
    var_99 = ret.quantile(0.01)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(ret, bins=50, density=True, alpha=0.6, edgecolor='black')
    ret.plot.kde(ax=ax, color='black', linewidth=2)

    ax.axvline(var_95, color='orange', linewidth=2, linestyle='--',
               label=f'VaR 95%: {var_95*100:.3f}%')
    ax.axvline(var_99, color='red', linewidth=2, linestyle='--',
               label=f'VaR 99%: {var_99*100:.3f}%')

    # 꼬리 영역 색칠
    ax.axvspan(ret.min(), var_95, alpha=0.15, color='red')

    ax.set_title(f'{col} — 수익률 분포 & VaR')
    ax.set_xlabel('일간 수익률')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 6. 리스크-수익 산점도

In [ ]:
# 리스크(연환산 변동성) vs 수익(연환산 수익률) 산점도
scatter_data = []

for col in returns_df.columns:
    ret = returns_df[col]
    source_part = col.split('(')[-1].rstrip(')')
    ann_factor = 365 if source_part in CRYPTO_SOURCES else 252

    scatter_data.append({
        '자산': col,
        '연환산 변동성': ret.std() * np.sqrt(ann_factor) * 100,
        '연환산 수익률': ret.mean() * ann_factor * 100,
        '데이터 수': len(ret),
        'Max Drawdown': abs(max_dd[col]) * 100
    })

scatter_df = pd.DataFrame(scatter_data)

fig = px.scatter(scatter_df, x='연환산 변동성', y='연환산 수익률',
                 text='자산', size='데이터 수', color='Max Drawdown',
                 color_continuous_scale='Reds',
                 labels={'연환산 변동성': '연환산 변동성 (%)',
                         '연환산 수익률': '연환산 수익률 (%)'})

fig.update_traces(textposition='top center')

# 사분면 구분선
fig.add_hline(y=0, line_dash='dot', line_color='gray')
fig.add_vline(x=scatter_df['연환산 변동성'].median(), line_dash='dot', line_color='gray')

# 사분면 라벨
fig.add_annotation(x=scatter_df['연환산 변동성'].min(), y=scatter_df['연환산 수익률'].max(),
                   text='고수익/저위험', showarrow=False, font=dict(color='green', size=12))
fig.add_annotation(x=scatter_df['연환산 변동성'].max(), y=scatter_df['연환산 수익률'].max(),
                   text='고수익/고위험', showarrow=False, font=dict(color='orange', size=12))
fig.add_annotation(x=scatter_df['연환산 변동성'].min(), y=scatter_df['연환산 수익률'].min(),
                   text='저수익/저위험', showarrow=False, font=dict(color='blue', size=12))
fig.add_annotation(x=scatter_df['연환산 변동성'].max(), y=scatter_df['연환산 수익률'].min(),
                   text='저수익/고위험', showarrow=False, font=dict(color='red', size=12))

fig.update_layout(
    title='리스크-수익 산점도 (버블 크기 = 데이터 수, 색상 = Max Drawdown)',
    height=600
)
fig.show()

## 7. 종합 리스크 대시보드

In [ ]:
# 모든 지표를 한 테이블로
dashboard = []

for col in returns_df.columns:
    ret = returns_df[col]
    source_part = col.split('(')[-1].rstrip(')')
    ann_factor = 365 if source_part in CRYPTO_SOURCES else 252

    daily_mean = ret.mean()
    daily_std = ret.std()
    ann_return = daily_mean * ann_factor
    ann_vol = daily_std * np.sqrt(ann_factor)
    sharpe = ann_return / ann_vol if ann_vol > 0 else 0
    var_95 = ret.quantile(0.05)
    cvar_95 = ret[ret <= var_95].mean()

    # 최신 가격
    latest_price = prices_df[col].iloc[-1] if col in prices_df.columns else None
    total_return = (prices_df[col].iloc[-1] / prices_df[col].iloc[0] - 1) if col in prices_df.columns else None

    dashboard.append({
        '자산': col,
        '최신가': f'{latest_price:,.2f}' if latest_price else 'N/A',
        '총 수익률': f'{total_return*100:.2f}%' if total_return else 'N/A',
        '연환산 수익률': f'{ann_return*100:.2f}%',
        '연환산 변동성': f'{ann_vol*100:.2f}%',
        'Sharpe': f'{sharpe:.3f}',
        'Max DD': f'{max_dd[col]*100:.2f}%',
        'VaR 95%': f'{var_95*100:.3f}%',
        'CVaR 95%': f'{cvar_95*100:.3f}%',
        '첨도': f'{ret.kurtosis():.2f}',
        '왜도': f'{ret.skew():.2f}'
    })

dashboard_df = pd.DataFrame(dashboard)
dashboard_df.style.set_caption('종합 리스크-수익 대시보드')